# Markt versus markt + nieuws/X
Lokale, ordervrije evaluatie. Kies hieronder een run. Hoge accuracy op enkele uren bewijst niets. Modellen zijn bevroren; dit notebook traint niet opnieuw en doet geen API-calls. De 7/30-dagenmodellen blijven geblokkeerd zolang labels of splits ontbreken.

In [ ]:
import html
import json
import sqlite3
from pathlib import Path

from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
runs = sorted(
    (ROOT / "experiments/results").glob("content-ETH-EUR-*/report.json"),
    key=lambda p: p.stat().st_mtime,
)
assert runs, "Voer eerst python -m ai_trainer compare-content --market ETH-EUR uit."
REPORT = runs[-1]  # vervang voor een andere markt/run
report = json.loads(REPORT.read_text(encoding="utf-8"))
print(REPORT)
print(report["status"], "| beslisuren:", report["observation_windows"])
print("Splits:", report.get("validation_start"), report.get("test_start"))
for warning in report["limitations"]:
    print("-", warning)

In [ ]:
def table(headers, rows):
    def cells(values):
        return "".join("<td>" + html.escape(str(v)) + "</td>" for v in values)

    display(
        HTML(
            "<table><tr>"
            + cells(headers)
            + "</tr>"
            + "".join("<tr>" + cells(row) + "</tr>" for row in rows)
            + "</table>"
        )
    )


metrics = []
for horizon, result in report["horizons"].items():
    for model, values in result.get("models", {}).items():
        m = values["test"]
        p = m["paper"]
        metrics.append(
            [
                horizon,
                model,
                result["partition_counts"],
                round(m["rmse"], 5),
                round(m["zero_return_rmse"], 5),
                p["trades"],
                round(100 * p["net_return"], 3),
                round(100 * p["max_drawdown"], 3),
            ]
        )
table(
    ["Horizon", "Model", "Train/val/test", "RMSE", "Nul-RMSE", "Trades", "Netto %", "Drawdown %"],
    metrics,
)
print(
    "Fees: 0.25% per zijde; spread/slippage: 0.10% per zijde. Allocatie: 10%, maximaal één positie."
)

In [ ]:
HORIZON = "1h"
prediction_file = REPORT.parent / f"predictions_market_text_{HORIZON}.jsonl"
predictions = (
    [json.loads(line) for line in prediction_file.read_text(encoding="utf-8").splitlines()]
    if prediction_file.exists()
    else []
)
table(
    ["Beslismoment", "Voorspeld %", "Werkelijk %", "Berichten (ingekort)"],
    [
        [
            r["decision_at"],
            round(100 * r["predicted"], 3),
            round(100 * r["actual"], 3),
            r["text"][:220],
        ]
        for r in predictions
    ],
)
if len(predictions) > 1:
    values = [100 * r[k] for r in predictions for k in ("predicted", "actual")]
    lo, hi = min(values), max(values)
    span = max(hi - lo, 0.001)

    def points(key):
        return " ".join(
            f"{20 + i * 760 / (len(predictions) - 1):.1f},"
            f"{220 - (100 * r[key] - lo) * 190 / span:.1f}"
            for i, r in enumerate(predictions)
        )

    display(
        HTML(
            "<p>Blauw: voorspeld; oranje: werkelijk. X: testmomenten; Y: rendement.</p>"
            "<svg width=800 height=250>"
            + "".join(
                f'<polyline fill=none stroke={color} stroke-width=2 points="{points(key)}" />'
                for key, color in [("predicted", "steelblue"), ("actual", "darkorange")]
            )
            + "</svg>"
        )
    )

In [ ]:
journal = ROOT / "experiments/results/content-shadow.db"
if journal.exists():
    with sqlite3.connect(f"file:{journal.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute(
            "SELECT substr(p.model_id,1,10), p.market, p.recorded_at, p.entry_at, "
            "p.prediction, o.actual FROM predictions p LEFT JOIN outcomes o "
            "USING(model_id,market,entry_at) ORDER BY p.recorded_at DESC LIMIT 30"
        ).fetchall()
    table(
        [
            "Model",
            "Markt",
            "Vastgelegd",
            "Geplande ingang",
            "Voorspelling",
            "Uitkomst (None = nog onbekend)",
        ],
        rows,
    )
else:
    print("Nog geen vooruit-test gestart.")

## Operationele controle
Live, alleen-lezen overzicht per model. Nog niet verstreken is geen ontbrekende uitkomst. 'Vervallen/open' vraagt controle van candles en collectorvertraging; het is geen verlies. Er wordt geen winnaar geselecteerd.

In [ ]:
from datetime import UTC, datetime

from ai_trainer.infrastructure.models.shadow_status import journal_status

if journal.exists():
    audit = journal_status(journal, datetime.now(UTC))
    print("Database:", audit["integrity"], "| afwijkingen:", audit["violations"])
    table(
        [
            "Model",
            "Markt",
            "Aantal",
            "Afgerond",
            "Nog niet verstreken",
            "Vervallen/open",
            "MAE",
            "RMSE",
            "Volgende uitkomst UTC",
        ],
        [
            [
                m["model_id"][:10],
                m["market"],
                m["predictions"],
                m["matured"],
                m["pending_not_due"],
                m["due_unsettled"],
                m["mae"],
                m["rmse"],
                m["next_label_available_at"],
            ]
            for m in audit["models"]
        ],
    )
for prefix in ("public-collector", "content-shadow"):
    files = list((ROOT / "logs").glob(prefix + "*.log"))
    if files:
        latest = max(files, key=lambda p: p.stat().st_mtime)
        print("Laatste log:", latest.name, datetime.fromtimestamp(latest.stat().st_mtime, UTC))
        print("\n".join(latest.read_text(encoding="utf-8", errors="replace").splitlines()[-3:]))